# Interactive simulation checks: facility choice

Verifies the correlated propensities driving ANC / in-facility delivery / LBWSG, the
believed-vs-true preterm confusion matrix against the facility-choice validation
targets, and gestational-age estimation error by ultrasound type. Ported from the
research portfolio VnV notebook `model_16.3_interactive_simulation_facility_choice`;
updated to the current Engine (`vivarium.engine`) API and converted to assert-based checks.

In [1]:
import warnings
warnings.simplefilter(action="ignore", category=FutureWarning)

import numpy as np
import pandas as pd
from pathlib import Path

import vivarium_gates_mncnh
from vivarium.artifact import Artifact
from vivarium.interactive import Simulation
from vivarium.engine.framework.configuration import build_model_specification

In [2]:
!pip list | grep vivarium

pytest-vivarium                          0.3.1
vivarium-artifact                        1.1.1
vivarium-build-utils                     4.11.0
vivarium-cluster-tools                   4.8.0
vivarium-config-tree                     5.2.1
vivarium-dependencies                    1.3.1
vivarium-engine                          6.0.1
vivarium-fuzzy-checker                   0.5.1
vivarium_gates_mncnh                     41.3.dev26+g667ae9674.d20261110 /mnt/share/homes/hjafari/repos/vivarium_gates_mncnh-mic-7717
vivarium-gbd-mapping                     6.0.7
vivarium-interactive                     1.0.1
vivarium-public-health                   6.8.0
vivarium-risk-distributions              3.2.1



[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: pip install --upgrade pip


In [3]:
spec_path = Path(vivarium_gates_mncnh.__file__).parent / "model_specifications/model_spec.yaml"
model_spec = build_model_specification(spec_path)
del model_spec.configuration.observers
model_spec.configuration.population.population_size = 20_000 * 10
# Facility-choice / ultrasound VnV scenario.
model_spec.configuration.intervention.scenario = "ultrasound_vv"

artifact_path = model_spec.configuration.input_data.artifact_path
art = Artifact(artifact_path)
draw = "draw_" + str(model_spec.configuration.input_data.input_draw_number)
# validation-target column follows the artifact's location, e.g. 'ethiopia.hdf' -> 'Ethiopia'
location = Path(artifact_path).stem.capitalize()
artifact_path, draw, location

('/mnt/team/simulation_science/pub/models/vivarium_gates_mncnh/artifacts/emotive_with_live_hemoglobin_sd/ethiopia.hdf',
 'draw_60',
 'Ethiopia')

In [4]:
sim = Simulation(model_spec)

2026-11-12 13:32:58.321 | 0:00:12.230250 | WARNING  | simulation_1-values_manager:_warn_if_overriding_resources:488 - Conflicting information for birth_outcome_probabilities. Ignoring 'required_resources' since the `source` is of type <class 'vivarium.engine.framework.lookup.table.LookupTable'> and we can infer the required resources directly.


2026-11-12 13:33:00.936 | 0:00:14.845016 | WARNING  | simulation_1-values_manager:_warn_if_overriding_resources:488 - Conflicting information for lbwsg_paf_on_all_causes.all_cause_mortality_risk.paf. Ignoring 'required_resources' since the `source` is of type <class 'vivarium.engine.framework.lookup.table.LookupTable'> and we can infer the required resources directly.


2026-11-12 13:33:00.980 | 0:00:14.889203 | WARNING  | simulation_1-values_manager:_warn_if_overriding_resources:488 - Conflicting information for lbwsg_paf_on_neonatal_sepsis_and_other_neonatal_infections.cause_specific_mortality_risk.paf. Ignoring 'required_resources' since the `source` is of type <class 'vivarium.engine.framework.lookup.table.LookupTable'> and we can infer the required resources directly.


2026-11-12 13:33:01.007 | 0:00:14.915953 | WARNING  | simulation_1-values_manager:_warn_if_overriding_resources:488 - Conflicting information for lbwsg_paf_on_neonatal_preterm_birth_with_rds.cause_specific_mortality_risk.paf. Ignoring 'required_resources' since the `source` is of type <class 'vivarium.engine.framework.lookup.table.LookupTable'> and we can infer the required resources directly.


2026-11-12 13:33:01.032 | 0:00:14.940904 | WARNING  | simulation_1-values_manager:_warn_if_overriding_resources:488 - Conflicting information for lbwsg_paf_on_neonatal_preterm_birth_without_rds.cause_specific_mortality_risk.paf. Ignoring 'required_resources' since the `source` is of type <class 'vivarium.engine.framework.lookup.table.LookupTable'> and we can infer the required resources directly.


2026-11-12 13:33:01.057 | 0:00:14.965711 | WARNING  | simulation_1-values_manager:_warn_if_overriding_resources:488 - Conflicting information for lbwsg_paf_on_neonatal_encephalopathy_due_to_birth_asphyxia_and_trauma.cause_specific_mortality_risk.paf. Ignoring 'required_resources' since the `source` is of type <class 'vivarium.engine.framework.lookup.table.LookupTable'> and we can infer the required resources directly.


... (54 outputs left out)

2026-11-12 13:33:08.927 | 0:00:22.835792 | WARNING  | simulation_1-lookup_table_manager:on_post_setup:84 - Component 'non_log_linear_risk_effect.hemoglobin_on_cause.postpartum_hemorrhage.incidence_risk' configured, but didn't build lookup table 'population_attributable_fraction' during setup.


2026-11-12 13:33:08.927 | 0:00:22.836225 | WARNING  | simulation_1-lookup_table_manager:on_post_setup:84 - Component 'non_log_linear_risk_effect.hemoglobin_on_cause.maternal_sepsis_and_other_maternal_infections.incidence_risk' configured, but didn't build lookup table 'population_attributable_fraction' during setup.


2026-11-12 13:33:08.928 | 0:00:22.836660 | WARNING  | simulation_1-lookup_table_manager:on_post_setup:84 - Component 'non_log_linear_risk_effect.hemoglobin_on_cause.postpartum_depression.incidence_risk' configured, but didn't build lookup table 'population_attributable_fraction' during setup.


2026-11-12 13:33:08.928 | 0:00:22.837109 | WARNING  | simulation_1-lookup_table_manager:on_post_setup:84 - Component 'non_log_linear_risk_effect.hemoglobin_on_cause.neonatal_sepsis_and_other_neonatal_infections.cause_specific_mortality_risk' configured, but didn't build lookup table 'population_attributable_fraction' during setup.


In [5]:
# Step up to and through the delivery_facility event.
get_event_name = sim._builder.time.simulation_event_name()
while get_event_name() != "delivery_facility":
    sim.step()
sim.step()  # complete the delivery_facility event

In [6]:
PROP_ANC = "antenatal_care.correlated_propensity"
PROP_LBWSG = "risk_factor.low_birth_weight_and_short_gestation.correlated_propensity"
PROP_IFD = "delivery_facility.correlated_propensity"

# get_population accepts state columns and pipeline/attribute names together.
COLS = ["anc_attendance", "delivery_facility_type", "gestational_age.exposure",
        "stated_gestational_age", "pregnancy_outcome", "ultrasound_type"]
df = sim.get_population(COLS + [PROP_ANC, PROP_LBWSG, PROP_IFD])
df["anc1"] = df.anc_attendance != "none"
df["ifd"] = df.delivery_facility_type.isin(["BEmONC", "CEmONC"])
df["preterm"] = df["gestational_age.exposure"] < 37
df["believed_preterm"] = df.stated_gestational_age < 37
df = df.loc[df.pregnancy_outcome != "partial_term"]  # checks are on births
df[["anc_attendance", "delivery_facility_type", "ultrasound_type", "anc1", "ifd", "preterm", "believed_preterm"]].head()

,anc_attendance,delivery_facility_type,ultrasound_type,anc1,ifd,preterm,believed_preterm
0,first_trimester_and_later_pregnancy,CEmONC,AI_assisted,True,True,False,False
4,later_pregnancy_only,home,standard,True,False,True,False
5,first_trimester_and_later_pregnancy,home,standard,True,False,False,True
6,none,home,no_ultrasound,False,False,False,False
7,later_pregnancy_only,CEmONC,standard,True,True,True,True


## Correlated propensities are ordered as expected

In [7]:
# Higher ANC propensity -> attends ANC; higher facility propensity -> in-facility delivery;
# LOWER LBWSG propensity -> more likely preterm.
assert df.groupby("anc1")[PROP_ANC].mean().loc[True] > df.groupby("anc1")[PROP_ANC].mean().loc[False], \
    "ANC attendees do not have higher ANC propensity"
assert df.groupby("ifd")[PROP_IFD].mean().loc[True] > df.groupby("ifd")[PROP_IFD].mean().loc[False], \
    "in-facility deliveries do not have higher facility propensity"
assert df.groupby("preterm")[PROP_LBWSG].mean().loc[True] < df.groupby("preterm")[PROP_LBWSG].mean().loc[False], \
    "preterm births do not have lower LBWSG propensity"

## Propensity correlations match the facility-choice targets

In [8]:
# Target correlations (Ethiopia): ~0.63 between ANC and IFD propensity, ~0.2 otherwise.
corr = df[[PROP_ANC, PROP_LBWSG, PROP_IFD]].corr()
assert np.isclose(corr.loc[PROP_ANC, PROP_IFD], 0.63, atol=0.1), \
    f"ANC-IFD propensity corr {corr.loc[PROP_ANC, PROP_IFD]:.3f} far from ~0.63"
assert np.isclose(corr.loc[PROP_ANC, PROP_LBWSG], 0.2, atol=0.1), \
    f"ANC-LBWSG propensity corr {corr.loc[PROP_ANC, PROP_LBWSG]:.3f} far from ~0.2"
assert np.isclose(corr.loc[PROP_LBWSG, PROP_IFD], 0.2, atol=0.1), \
    f"LBWSG-IFD propensity corr {corr.loc[PROP_LBWSG, PROP_IFD]:.3f} far from ~0.2"

## Believed-vs-true preterm matches the validation targets

In [9]:
targets = pd.read_csv(
    Path(vivarium_gates_mncnh.__file__).parent / "data/facility_choice/facility_choice_validation_targets.csv"
)
def target(prob_of):
    return targets.loc[targets.probability_of == prob_of, location].values[0]

# P(believed_preterm | true preterm-status), pooled over ultrasound type.
obs = df.groupby("preterm").believed_preterm.value_counts(normalize=True)
assert np.isclose(obs.loc[(True, True)], target("believed_preterm_given_preterm"), atol=0.05), \
    "P(believed preterm | preterm) off target"
assert np.isclose(obs.loc[(False, False)], target("believed_term_given_term"), atol=0.05), \
    "P(believed term | term) off target"

In [10]:
# Same, stratified by ultrasound type (targets exist for no_ultrasound and standard only).
tri = df.groupby(["ultrasound_type", "preterm"]).believed_preterm.value_counts(normalize=True)
suffix = {"no_ultrasound": "no_ultrasound", "standard": "standard_ultrasound"}
for us in ["no_ultrasound", "standard"]:
    for pt in [True, False]:
        pt_word = "preterm" if pt else "term"
        key = f"believed_preterm_given_{pt_word}_and_{suffix[us]}"
        obs_val = tri.loc[(us, pt, True)]
        assert np.isclose(obs_val, target(key), atol=0.05), \
            f"{key}: observed {obs_val:.3f} vs target {target(key):.3f}"

## Gestational-age estimation error by ultrasound type

In [11]:
# ga_error in days; unbiased (~0 mean) for each type, and accuracy improves
# AI-assisted < standard < no-ultrasound (smaller spread = more accurate).
df["ga_error"] = (df.stated_gestational_age - df["gestational_age.exposure"]) * 7
means = df.groupby("ultrasound_type").ga_error.mean()
stds = df.groupby("ultrasound_type").ga_error.std()
for us in ["no_ultrasound", "standard", "AI_assisted"]:
    assert abs(means.loc[us]) < 1.0, f"{us} GA-error mean {means.loc[us]:.3f} not ~0"
assert stds.loc["AI_assisted"] < stds.loc["standard"] < stds.loc["no_ultrasound"], \
    f"GA-error spread not ordered AI < standard < none: {stds.to_dict()}"